In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import hashlib
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from models.autoencoder_classifier import *
from final_pipline.similarity_latent import *
from final_pipline.latent_similarty_funcs import *
import torch
from config import *
from numpy.linalg import norm

In [3]:
cs = build_balanced_cs_loaders_from_h5(
    h5_path=xrd_dataset,
    per_class_cs=105000,   
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)
print(cs["counts"])
print(cs["sizes"])

{'triclinic': 105000, 'monoclinic': 105000, 'orthorhombic': 105000, 'tetragonal': 105000, 'trigonal': 105000, 'hexagonal': 105000, 'cubic': 105000}
{'train': 588000, 'val': 73500, 'test': 73500}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate model EXACTLY the same way
model = DeepConvAutoencoderClassifier(
    input_length=cs["input_len"],
    latent_dim=64,
    cls_dim=128,
    num_classes=cs["num_classes"],
    use_projection_head=True
).to(device)

# Load state_dict safely
state_dict = torch.load(CS_Cls, map_location=device, weights_only=True)

model.load_state_dict(state_dict)

model.eval()

DeepConvAutoencoderClassifierV2(
  (encoder): Sequential(
    (0): ResBlock1D(
      (conv1): Conv1d(1, 32, kernel_size=(15,), stride=(2,), padding=(7,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(32, 32, kernel_size=(15,), stride=(1,), padding=(7,))
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (skip): Conv1d(1, 32, kernel_size=(1,), stride=(2,))
      (act): GELU(approximate='none')
      (attn): ChannelAttention(
        (net): Sequential(
          (0): AdaptiveAvgPool1d(output_size=1)
          (1): Conv1d(32, 2, kernel_size=(1,), stride=(1,))
          (2): GELU(approximate='none')
          (3): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
          (4): Sigmoid()
        )
      )
    )
    (1): ResBlock1D(
      (conv1): Conv1d(32, 64, kernel_size=(11,), stride=(2,), padding=(5,))
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_r

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_latents, train_labels, train_indices = extract_latent_vectors(
    model,
    cs,
    device,
    loader_name="train_loader"
)

print("Train latents shape:", train_latents.shape)
print("Train labels shape:", train_labels.shape)
print("Train indices shape:", train_indices.shape)


In [5]:
train_path = os.path.join(CS_LATENT, "xrd_train_latents.npz")
np.savez_compressed(
    train_path,
    latents=train_latents,
    labels=train_labels,
    indices=train_indices,
    class_names=cs["class_names"]
)

In [7]:
train_path = os.path.join(CS_LATENT, "xrd_train_latents.npz")
data = np.load(train_path, allow_pickle=True)
train_latents = data["latents"]
train_labels  = data["labels"]
train_indices = data["indices"]
class_names   = data["class_names"]

print("Loaded shapes:")
print("Latents:", train_latents.shape)
print("Labels:", train_labels.shape)
print("Indices:", train_indices.shape)
print("Classes:", class_names)


Loaded shapes:
Latents: (587776, 128)
Labels: (587776,)
Indices: (587776,)
Classes: ['triclinic' 'monoclinic' 'orthorhombic' 'tetragonal' 'trigonal'
 'hexagonal' 'cubic']
